<a href="https://colab.research.google.com/github/Mrunald3/OpenFDA/blob/main/fda_recall_rag.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# FDA Drug Recall Assistant (RAG)
A retrieval-augmented assistant that answers questions about FDA drug recalls and cites the source records.

**Pipeline:** openFDA enforcement data -> one chunk per recall -> Chroma vector store -> top-k retrieval -> Claude answers with citations -> evaluation on 20 test questions.


## 1. Install and set up

In [ ]:
!pip install -q chromadb anthropic requests pandas

In [ ]:
import os, json, random, re
import requests, pandas as pd
import chromadb
from anthropic import Anthropic
from google.colab import userdata

client = Anthropic(api_key=userdata.get('ANTHROPIC_API_KEY'))
MODEL = "claude-sonnet-5-5"   # change here if you want a different Claude model
random.seed(42)
print("Setup OK")

## 2. Pull FDA drug recall data (openFDA)

In [ ]:
N_RECORDS = 100
url = "https://api.fda.gov/drug/enforcement.json"
params = {"search": "report_date:[20240101 TO 20251231]", "limit": 200}
resp = requests.get(url, params=params, timeout=30)
resp.raise_for_status()
results = resp.json()["results"]
print("Fetched", len(results), "records")

KEEP = ["recall_number", "product_description", "recalling_firm", "reason_for_recall",
        "classification", "status", "distribution_pattern", "recall_initiation_date"]
rows, seen = [], set()
for r in results:
    rn = r.get("recall_number")
    if not rn or rn in seen:
        continue
    if not all(r.get(k) for k in KEEP):
        continue
    seen.add(rn)
    rows.append({k: r[k] for k in KEEP})
    if len(rows) == N_RECORDS:
        break

df = pd.DataFrame(rows)
df.to_json("recalls.json", orient="records", indent=2)
print("Kept", len(df), "complete, unique records")
df.head(3)

## 3. Chunk and embed into Chroma
Each recall is short, so one record = one chunk. Chroma's default embedding model (all-MiniLM-L6-v2) runs locally, so no extra API key is needed.

In [ ]:
def to_text(r):
    return (f"Recall {r['recall_number']}. Product: {r['product_description']}. "
            f"Recalling firm: {r['recalling_firm']}. Reason: {r['reason_for_recall']}. "
            f"Classification: {r['classification']}. Status: {r['status']}. "
            f"Distribution: {r['distribution_pattern']}. Initiated: {r['recall_initiation_date']}.")

chroma = chromadb.Client()
try:
    chroma.delete_collection("recalls")
except Exception:
    pass
col = chroma.create_collection("recalls")
col.add(
    ids=df["recall_number"].tolist(),
    documents=[to_text(r) for r in rows],
    metadatas=[{"recalling_firm": r["recalling_firm"], "classification": r["classification"]} for r in rows],
)
print("Indexed", col.count(), "recalls")

## 4. The RAG function

In [ ]:
SYSTEM = (
    "You answer questions about FDA drug recalls using ONLY the recall records provided. "
    "Cite the recall number in square brackets after each fact, like [D-1234-2025]. "
    "If the records do not contain the answer, say you cannot find it in the provided records. "
    "Keep answers short."
)

def retrieve(question, k=5):
    out = col.query(query_texts=[question], n_results=k)
    return out["ids"][0], out["documents"][0]

def ask(question, k=5):
    ids, docs = retrieve(question, k)
    context = "\n\n".join(docs)
    msg = client.messages.create(
        model=MODEL, max_tokens=400, system=SYSTEM,
        messages=[{"role": "user", "content": f"Recall records:\n{context}\n\nQuestion: {question}"}],
    )
    return msg.content[0].text, ids

answer, ids = ask("Which recalls were Class I and why?")
print(answer)
print("\nRetrieved:", ids)

Try a few of your own questions here:

In [ ]:
for q in ["Which companies recalled products for contamination?",
          "Are there any recalls related to labeling errors?",
          "What was recalled for failing dissolution specifications?"]:
    a, ids = ask(q)
    print("Q:", q); print(a); print("Sources:", ids); print("-" * 60)

## 5. Evaluation
Builds 20 test questions from the data itself (so the right answer is known): 7 classification, 7 recalling firm, 6 reason. Measures:
- **Retrieval hit rate:** was the correct recall in the top-k results?
- **Answer correctness:** judged by Claude against the known answer.
- **Citation rate:** did the answer cite a recall that was actually retrieved?

In [ ]:
def short(p, n=90):
    p = re.sub(r"\s+", " ", p).strip()
    return p[:n]

sample = df.sample(20, random_state=42).reset_index(drop=True)
tests = []
for i, r in sample.iterrows():
    prod, firm = short(r["product_description"]), r["recalling_firm"]
    if i < 7:
        q = f"What was the recall classification for {prod}, recalled by {firm}?"
        exp = r["classification"]
    elif i < 14:
        q = f"Which company recalled this product: {prod}?"
        exp = firm
    else:
        q = f"Why was {prod}, from {firm}, recalled?"
        exp = r["reason_for_recall"]
    tests.append({"question": q, "expected": exp, "recall_number": r["recall_number"]})

json.dump(tests, open("questions.json", "w"), indent=2)
print(len(tests), "test questions saved")
tests[0]

In [ ]:
def judge(question, expected, answer):
    msg = client.messages.create(
        model=MODEL, max_tokens=5,
        messages=[{"role": "user", "content":
            f"Question: {question}\nExpected answer: {expected}\nGiven answer: {answer}\n"
            "Does the given answer convey the expected answer? Reply only YES or NO."}],
    )
    return msg.content[0].text.strip().upper().startswith("YES")

def evaluate(k):
    rows_out = []
    for t in tests:
        answer, ids = ask(t["question"], k)
        cited = re.findall(r"\[([A-Z]-\d+-\d{4})\]", answer)
        rows_out.append({
            "question": t["question"],
            "hit": t["recall_number"] in ids,
            "correct": judge(t["question"], t["expected"], answer),
            "cited_retrieved": any(c in ids for c in cited),
            "answer": answer,
            "expected": t["expected"],
        })
    return pd.DataFrame(rows_out)

summary = {}
details = {}
for k in (3, 5):
    d = evaluate(k)
    details[k] = d
    summary[f"top-{k}"] = {
        "retrieval hit rate": round(d["hit"].mean() * 100),
        "answer correct %": round(d["correct"].mean() * 100),
        "citation rate %": round(d["cited_retrieved"].mean() * 100),
    }
pd.DataFrame(summary)

## 6. Look at failures (use one as your 'what went wrong' story)

In [ ]:
d = details[5]
fails = d[~d["correct"]]
print(len(fails), "incorrect answers at top-5")
for _, r in fails.head(5).iterrows():
    print("Q:", r["question"]); print("Expected:", r["expected"]); print("Got:", r["answer"]); print("Retrieval hit:", r["hit"]); print("-" * 60)

## 7. Save your results
Run the next cell, download `recalls.json`, `questions.json` and this notebook, and put them in a GitHub repo with a short README that includes the results table above. Use only your real numbers on your resume.

In [ ]:
pd.DataFrame(summary).to_csv("eval_results.csv")
print(pd.DataFrame(summary))